# Custom evaluator registration

Registers `evaluators/custom_compliance_phrase.py` into the project evaluator catalog.
Source: https://learn.microsoft.com/azure/foundry/concepts/evaluation-evaluators/custom-evaluators

In [ ]:
import json as _json
import os, pathlib, json
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Discover the active azd env dynamically: prefer AZURE_ENV_NAME, else read
# the 'defaultEnvironment' from agent/.azure/config.json.
AGENT_DIR = pathlib.Path('..') / 'agent'
_azd_base = AGENT_DIR / '.azure'
_env_name = os.environ.get('AZURE_ENV_NAME', '')
if not _env_name:
    _config = _azd_base / 'config.json'
    if _config.exists():
        try:
            _env_name = _json.loads(_config.read_text()).get('defaultEnvironment', '')
        except Exception:
            _env_name = ''
if _env_name and (_azd_base / _env_name / '.env').exists():
    AZD_ENV = _azd_base / _env_name / '.env'
else:
    AZD_ENV = next(
        (p / '.env' for p in sorted(_azd_base.iterdir()) if p.is_dir() and (p / '.env').exists()),
        _azd_base / 'default' / '.env',
    )
if AZD_ENV.exists():
    load_dotenv(AZD_ENV)
load_dotenv()
endpoint = os.environ['AZURE_AI_PROJECT_ENDPOINT']
project = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
print('azd env: ', _env_name or '(fallback)')
print('endpoint:', endpoint)

In [ ]:
# Approach: package the evaluator as a callable and register via the
# evaluator catalog REST/SDK. The exact SDK surface depends on
# azure-ai-projects version. We use the catalog endpoint directly.
import sys
sys.path.insert(0, str(pathlib.Path('..') / 'evaluators'))
import custom_compliance_phrase as cce
print('evaluator module:', cce)
print('sample:', cce.evaluate(response='This response is for informational purposes only. Hello.'))
print('sample:', cce.evaluate(response='Hello.'))

In [ ]:
# Register via project.beta.evaluators.create_version() (current SDK surface).
import yaml
from azure.ai.projects.models import (
    CodeBasedEvaluatorDefinition,
    EvaluatorCategory,
    EvaluatorDefinitionType,
    EvaluatorType,
    EvaluatorVersion,
)

manifest_path = pathlib.Path('..') / 'evaluators' / 'custom_compliance_phrase.yaml'
manifest = yaml.safe_load(manifest_path.read_text())
code = (pathlib.Path('..') / 'evaluators' / 'custom_compliance_phrase.py').read_text()

evaluator_version = EvaluatorVersion(
    display_name=manifest.get('display_name', manifest['name']),
    description=manifest.get('description'),
    evaluator_type=EvaluatorType.CUSTOM,
    categories=[EvaluatorCategory.QUALITY],
    definition=CodeBasedEvaluatorDefinition(
        type=EvaluatorDefinitionType.CODE,
        code_text=code,
        entry_point=manifest['entry_point'],
    ),
    metadata={'module': 'custom_compliance_phrase'},
    tags={'demo': 'agent-observability'},
)

try:
    created = project.beta.evaluators.create_version(
        name=manifest['name'], evaluator_version=evaluator_version
    )
    catalog_id = getattr(created, 'id', None) or getattr(created, 'name', None)
    print(f'Registered: name={created.name} version={created.version} id={catalog_id}')
except Exception as e:
    print(f'Registration failed, falling back: {type(e).__name__}: {e}')
    catalog_id = f"manifest:{manifest['name']}:{manifest['version']}"

pathlib.Path('../artifacts').mkdir(exist_ok=True)
pathlib.Path('../artifacts/custom-evaluator.json').write_text(
    json.dumps({'catalog_id': catalog_id, 'name': manifest['name'], 'version': manifest.get('version', '1')}, indent=2, default=str)
)
print('Saved artifacts/custom-evaluator.json')